# Example: Coherence Analysis and Signal Quality Assessment

This example demonstrates how to use the squared coherence $\gamma^2(\omega)$ to
assess the quality of a frequency response estimate. Coherence near 1
indicates a reliable estimate; near 0 means noise dominates.
Coherence is only available for SISO systems estimated with `sid.freq_bt`
or `sid.freq_btfdr`.

Translated from `exampleCoherence.m`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter
import sid

%matplotlib inline

## Generate data: ARMA system with colored noise

$G(z) = (1 + 0.5\,z^{-1}) / (1 - 0.8\,z^{-1})$

Noise is colored: $v(t) = e(t) / (1 - 0.6\,z^{-1})$, so coherence varies
across frequency.

In [ ]:
rng = np.random.default_rng(3)

N = 2000
u = rng.standard_normal(N)
y_clean = lfilter([1, 0.5], [1, -0.8], u)
e = rng.standard_normal(N)
v = 0.5 * lfilter([1], [1, -0.6], e)       # colored noise
y = y_clean + v

## Estimate with `freq_bt`

In [ ]:
result = sid.freq_bt(y, u, window_size=40)

## Plot Bode magnitude and coherence together

In [ ]:
w = result.frequency

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

ax1.semilogx(w, 20 * np.log10(np.abs(result.response)), 'b')
ax1.set_ylabel('Magnitude (dB)')
ax1.set_title('Frequency Response Estimate')
ax1.grid(True)

ax2.semilogx(w, result.coherence, 'b')
ax2.axhline(0.9, color='g', ls='--', label=r'$\gamma^2 = 0.9$')
ax2.axhline(0.5, color='r', ls='--', label=r'$\gamma^2 = 0.5$')
ax2.set_ylabel(r'Coherence $\gamma^2$')
ax2.set_xlabel('Frequency (rad/sample)')
ax2.set_title('Squared Coherence')
ax2.legend(loc='lower left')
ax2.set_ylim([0, 1])
ax2.grid(True)

plt.tight_layout()
plt.show()

## Confidence bands reflect coherence

Where coherence is high, uncertainty is low (narrow bands).
The `confidence` option controls the number of standard deviations.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sid.bode_plot(result, confidence=2, ax=(axes[0, 0], axes[1, 0]))
axes[0, 0].set_title(r'$2\sigma$ Confidence Bands')

sid.bode_plot(result, confidence=3, ax=(axes[0, 1], axes[1, 1]))
axes[0, 1].set_title(r'$3\sigma$ Confidence Bands')

plt.tight_layout()
plt.show()

## High-noise vs low-noise comparison

More noise means lower coherence across all frequencies.

In [ ]:
rng2 = np.random.default_rng(3)

y_low  = y_clean + 0.1 * lfilter([1], [1, -0.6], rng2.standard_normal(N))
y_high = y_clean + 2.0 * lfilter([1], [1, -0.6], rng2.standard_normal(N))

r_low  = sid.freq_bt(y_low,  u, window_size=40)
r_high = sid.freq_bt(y_high, u, window_size=40)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(w, r_low.coherence,  'b', label=r'Low noise ($\sigma=0.1$)')
ax.semilogx(w, r_high.coherence, 'r', label=r'High noise ($\sigma=2.0$)')
ax.set_ylabel(r'Coherence $\gamma^2$')
ax.set_xlabel('Frequency (rad/sample)')
ax.set_title('Effect of Noise Level on Coherence')
ax.legend(loc='lower left')
ax.set_ylim([0, 1])
ax.grid(True)
plt.tight_layout()
plt.show()

## Note: ETFE does not provide coherence

`sid.freq_etfe` returns `coherence = None` because there is no windowed
cross-spectral estimate in the FFT-ratio approach.

In [ ]:
result_etfe = sid.freq_etfe(y, u)
print(f'ETFE coherence is None: {result_etfe.coherence is None}')